# ablation experiments
- baseline
- no crop
- GAN
- doubled sequence length

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from IPython.display import Audio, display

In [2]:
class CFG:
    SAMPLE_RATE = 16000
    SEGMENT     = 32000  
    N_FFT       = 512
    HOP         = 128
    BATCH_SIZE  = 8
    EPOCHS      = 2     
    LR          = 5e-4
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
    ROOT        = Path("/kaggle/input/datasets/muhmagdy/valentini-noisy")

In [3]:
class SpecUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self.block(1, 32)
        self.enc2 = self.block(32, 64)
        self.enc3 = self.block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = self.block(128, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = self.block(256, 128)
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = self.block(128, 64)
        self.up1  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = self.block(64, 32)
        self.out  = nn.Conv2d(32, 1, kernel_size=1)

    def block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        def up_cat(up_layer, dec_layer, feat, skip):
            d = up_layer(feat)
            dH, dW = skip.size(2) - d.size(2), skip.size(3) - d.size(3)
            d = F.pad(d, [dW//2, dW-dW//2, dH//2, dH-dH//2])
            return dec_layer(torch.cat([d, skip], dim=1))
        d3 = up_cat(self.up3, self.dec3, b,  e3)
        d2 = up_cat(self.up2, self.dec2, d3, e2)
        d1 = up_cat(self.up1, self.dec1, d2, e1)
        return F.softplus(self.out(d1))


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet = SpecUNet()
        self.register_buffer('window', torch.hann_window(CFG.N_FFT))

    def forward(self, noisy):
        spec         = torch.stft(noisy, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                                  window=self.window, return_complex=True)
        mag, phase   = spec.abs(), torch.angle(spec)
        mask         = self.unet(mag.unsqueeze(1)).squeeze(1)
        enhanced_mag = mask * mag
        enhanced     = torch.istft(torch.polar(enhanced_mag, phase),
                                   n_fft=CFG.N_FFT, hop_length=CFG.HOP, window=self.window)
        return enhanced, enhanced_mag


def si_sdr(pred, target, eps=1e-8):
    scale = (pred * target).sum(-1, keepdim=True) / (target.pow(2).sum(-1, keepdim=True) + eps)
    proj  = scale * target
    return (10 * torch.log10(proj.pow(2).sum(-1) / (((pred - proj).pow(2).sum(-1)) + eps) + eps)).mean()


def preview_audio(noisy, enhanced, clean, sr=CFG.SAMPLE_RATE):
    for label, t in [("Noisy", noisy), ("Enhanced", enhanced), ("Clean", clean)]:
        print(label); display(Audio(t.cpu().numpy(), rate=sr))


def preview_visuals(noisy, enhanced, clean, sr=CFG.SAMPLE_RATE):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    win = torch.hann_window(CFG.N_FFT)
    def db(a):
        s = torch.stft(a, n_fft=CFG.N_FFT, hop_length=CFG.HOP, window=win, return_complex=True)
        return librosa.amplitude_to_db(s.abs().numpy(), ref=np.max)
    for ax, a, t in zip(axes, [noisy, enhanced, clean], ["Noisy", "Enhanced", "Clean"]):
        librosa.display.specshow(db(a.cpu()), y_axis='linear', x_axis='time',
                                 sr=sr, hop_length=CFG.HOP, ax=ax, cmap='magma')
        ax.set_title(t)
    plt.tight_layout(); plt.show(); plt.close(fig)

In [4]:
def get_file_lists():
    noisy_train_dir = CFG.ROOT / "noisy_trainset_56spk_wav"
    clean_train_dir = CFG.ROOT / "clean_trainset_56spk_wav"
    noisy_test_dir  = CFG.ROOT / "noisy_testset_wav"
    clean_test_dir  = CFG.ROOT / "clean_testset_wav"

    train_noisy = sorted(noisy_train_dir.glob("*.wav"))
    train_clean = [clean_train_dir / f.name for f in train_noisy]
    train_noisy = [f for f in train_noisy if (clean_train_dir / f.name).exists()]
    train_clean = [clean_train_dir / f.name for f in train_noisy]

    val_noisy = sorted(noisy_test_dir.glob("*.wav"))
    val_clean = [clean_test_dir / f.name for f in val_noisy if (clean_test_dir / f.name).exists()]
    val_noisy = [f for f in val_noisy if (clean_test_dir / f.name).exists()]

    return train_noisy, train_clean, val_noisy, val_clean

train_noisy, train_clean, val_noisy, val_clean = get_file_lists()
print(f"Train: {len(train_noisy)} | Val: {len(val_noisy)}")

Train: 23075 | Val: 824


In [6]:
class ValentiniDataset(Dataset):
    def __init__(self, noisy_files, clean_files, segment=32000, train=True):
        self.noisy_files = noisy_files
        self.clean_files = clean_files
        self.segment = segment
        self.train = train

    def __getitem__(self, idx):
        noisy, _ = torchaudio.load(self.noisy_files[idx])
        clean, _ = torchaudio.load(self.clean_files[idx])

        if noisy.shape[-1] < self.segment:
            pad_amount = self.segment - noisy.shape[-1]
            noisy = torch.nn.functional.pad(noisy, (0, pad_amount))
            clean = torch.nn.functional.pad(clean, (0, pad_amount))

        if self.train:
            start = random.randint(0, noisy.shape[-1] - self.segment)
        else:
            start = 0

        noisy = noisy[..., start : start + self.segment]
        clean = clean[..., start : start + self.segment]

        return noisy.squeeze(0), clean.squeeze(0)

    def __len__(self):
        return len(self.noisy_files)

# baseline
this is the same as main model but 2 epochs, because all the others will be 2 epochs in order to finish training in time. we will compare relative drops/increases in accuracy rather than comparing to the main model which trained 15 epochs

In [7]:
base_train_ds = ValentiniDataset(train_noisy, train_clean, segment=CFG.SEGMENT, train=True)
base_val_ds   = ValentiniDataset(val_noisy,   val_clean,   segment=CFG.SEGMENT, train=False)
base_train_loader = DataLoader(base_train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,  num_workers=2)
base_val_loader   = DataLoader(base_val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2)

base_model = Denoiser().to(CFG.DEVICE)
base_opt   = torch.optim.Adam(base_model.parameters(), lr=CFG.LR)
window     = torch.hann_window(CFG.N_FFT).to(CFG.DEVICE)
base_history = {'train_loss': [], 'val_loss': [], 'val_sisdr': []}

print(f"Baseline - segment={CFG.SEGMENT} | batch={CFG.BATCH_SIZE} | epochs={CFG.EPOCHS}")

Baseline - segment=32000 | batch=8 | epochs=2


In [8]:
for epoch in range(CFG.EPOCHS):
    base_model.train()
    total_train = 0
    for noisy, clean in base_train_loader:
        noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
        base_opt.zero_grad()
        pred_wave, pred_mag = base_model(noisy)
        clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                               window=window, return_complex=True).abs()
        loss = F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)
        loss.backward()
        nn.utils.clip_grad_norm_(base_model.parameters(), 5.0)
        base_opt.step()
        total_train += loss.item()

    base_model.eval()
    total_val, total_sisdr = 0, 0
    with torch.no_grad():
        for noisy, clean in base_val_loader:
            noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
            pred_wave, pred_mag = base_model(noisy)
            clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                                   window=window, return_complex=True).abs()
            total_val   += (F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)).item()
            total_sisdr += si_sdr(pred_wave, clean).item()

    avg_tr = total_train / len(base_train_loader)
    avg_vl = total_val   / len(base_val_loader)
    avg_si = total_sisdr / len(base_val_loader)
    base_history['train_loss'].append(avg_tr)
    base_history['val_loss'].append(avg_vl)
    base_history['val_sisdr'].append(avg_si)
    print(f"[Baseline] Epoch {epoch+1:02d} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f} | SI-SDR: {avg_si:.2f} dB")

torch.save(base_model.state_dict(), "abl0_baseline.pth")
print("Saved abl0_baseline.pth")

[Baseline] Epoch 01 | Train: 0.0495 | Val: 0.0158 | SI-SDR: 10.31 dB
[Baseline] Epoch 02 | Train: 0.0423 | Val: 0.0149 | SI-SDR: 10.67 dB
Saved abl0_baseline.pth


## ablation 1 : remove cropping
we remove the cropping and adjust the padding accordingly, now the model will train on the full audio clip instead of 2 second segments.

In [9]:
class NoCropDataset(Dataset):
    """Returns the full waveform — no segment crop at all."""
    def __init__(self, noisy_files, clean_files, min_len=512):
        self.noisy_files = noisy_files
        self.clean_files = clean_files
        self.min_len = min_len

    def __getitem__(self, idx):
        noisy, _ = torchaudio.load(self.noisy_files[idx])
        clean, _ = torchaudio.load(self.clean_files[idx])

        if noisy.shape[-1] < self.min_len:
            pad = self.min_len - noisy.shape[-1]
            noisy = F.pad(noisy, (0, pad))
            clean = F.pad(clean, (0, pad))
        return noisy.squeeze(0), clean.squeeze(0)

    def __len__(self):
        return len(self.noisy_files)


def pad_collate(batch):
    noisy_list, clean_list = zip(*batch)
    max_len = max(x.shape[-1] for x in noisy_list)
    noisy_padded = torch.stack([F.pad(x, (0, max_len - x.shape[-1])) for x in noisy_list])
    clean_padded = torch.stack([F.pad(x, (0, max_len - x.shape[-1])) for x in clean_list])
    return noisy_padded, clean_padded


abl1_train_ds = NoCropDataset(train_noisy, train_clean)
abl1_val_ds   = NoCropDataset(val_noisy,   val_clean)

abl1_train_loader = DataLoader(abl1_train_ds, batch_size=2, shuffle=True,
                               num_workers=2, collate_fn=pad_collate)
abl1_val_loader   = DataLoader(abl1_val_ds,   batch_size=2, shuffle=False,
                               num_workers=2, collate_fn=pad_collate)

print(f"NoCrop — train batches: {len(abl1_train_loader)} | val batches: {len(abl1_val_loader)}")

NoCrop — train batches: 11538 | val batches: 412


In [ ]:
abl1_model = Denoiser().to(CFG.DEVICE)
abl1_opt   = torch.optim.Adam(abl1_model.parameters(), lr=CFG.LR)
window     = torch.hann_window(CFG.N_FFT).to(CFG.DEVICE)
abl1_history = {'train_loss': [], 'val_loss': [], 'val_sisdr': []}

for epoch in range(CFG.EPOCHS):
    abl1_model.train()
    total_train = 0
    for noisy, clean in abl1_train_loader:
        noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
        abl1_opt.zero_grad()
        pred_wave, pred_mag = abl1_model(noisy)
        clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                               window=window, return_complex=True).abs()
        min_len = min(pred_wave.shape[-1], clean.shape[-1])
        loss = F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave[..., :min_len], clean[..., :min_len])
        loss.backward()
        nn.utils.clip_grad_norm_(abl1_model.parameters(), 5.0)
        abl1_opt.step()
        total_train += loss.item()

    abl1_model.eval()
    total_val, total_sisdr = 0, 0
    with torch.no_grad():
        for noisy, clean in abl1_val_loader:
            noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
            pred_wave, pred_mag = abl1_model(noisy)
            clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                                   window=window, return_complex=True).abs()
            min_len = min(pred_wave.shape[-1], clean.shape[-1])
            total_val    += (F.l1_loss(pred_mag, clean_mag) +
                             0.2 * F.l1_loss(pred_wave[..., :min_len], clean[..., :min_len])).item()
            total_sisdr  += si_sdr(pred_wave[..., :min_len], clean[..., :min_len]).item()

    avg_tr = total_train / len(abl1_train_loader)
    avg_vl = total_val   / len(abl1_val_loader)
    avg_si = total_sisdr / len(abl1_val_loader)
    abl1_history['train_loss'].append(avg_tr)
    abl1_history['val_loss'].append(avg_vl)
    abl1_history['val_sisdr'].append(avg_si)
    print(f"[NoCrop] Epoch {epoch+1:02d} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f} | SI-SDR: {avg_si:.2f} dB")

torch.save(abl1_model.state_dict(), "abl1_nocrop.pth")
print("Saved abl1_nocrop.pth")

In [ ]:
abl1_model.eval()
sample_noisy, sample_clean = abl1_val_ds[0]
with torch.no_grad():
    enh, _ = abl1_model(sample_noisy.unsqueeze(0).to(CFG.DEVICE))
min_len = min(enh.shape[-1], sample_clean.shape[-1])
preview_audio(sample_noisy[:min_len], enh[0, :min_len], sample_clean[:min_len])
preview_visuals(sample_noisy[:min_len], enh[0, :min_len].cpu(), sample_clean[:min_len])

# ablation 2 : GAN
now we are seeing if the model is the issue. How will the accuracy change within the same no. of epochs if we change to a different model

In [ ]:
class WaveDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,  16, kernel_size=15, stride=4, padding=7),  nn.LeakyReLU(0.2),
            nn.Conv1d(16, 64, kernel_size=41, stride=4, padding=20, groups=4), nn.LeakyReLU(0.2),
            nn.Conv1d(64, 256, kernel_size=41, stride=4, padding=20, groups=16), nn.LeakyReLU(0.2),
            nn.Conv1d(256, 512, kernel_size=41, stride=4, padding=20, groups=64), nn.LeakyReLU(0.2),
            nn.Conv1d(512, 1, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1)).mean(dim=-1).squeeze(1) 


class ValentiniDataset(Dataset):
    def __init__(self, noisy_files, clean_files, segment=32000, train=True):
        self.noisy_files = noisy_files
        self.clean_files = clean_files
        self.segment = segment
        self.train = train

    def __getitem__(self, idx):
        noisy, _ = torchaudio.load(self.noisy_files[idx])
        clean, _ = torchaudio.load(self.clean_files[idx])
        if noisy.shape[-1] < self.segment:
            pad = self.segment - noisy.shape[-1]
            noisy = F.pad(noisy, (0, pad))
            clean = F.pad(clean, (0, pad))
        start = random.randint(0, noisy.shape[-1] - self.segment) if self.train else 0
        return noisy[..., start:start+self.segment].squeeze(0), clean[..., start:start+self.segment].squeeze(0)

    def __len__(self):
        return len(self.noisy_files)


abl2_train_ds = ValentiniDataset(train_noisy, train_clean, segment=CFG.SEGMENT, train=True)
abl2_val_ds   = ValentiniDataset(val_noisy,   val_clean,   segment=CFG.SEGMENT, train=False)
abl2_train_loader = DataLoader(abl2_train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,  num_workers=2)
abl2_val_loader   = DataLoader(abl2_val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2)

abl2_gen  = Denoiser().to(CFG.DEVICE)
abl2_disc = WaveDiscriminator().to(CFG.DEVICE)
abl2_gen_opt  = torch.optim.Adam(abl2_gen.parameters(),  lr=CFG.LR,    betas=(0.5, 0.9))
abl2_disc_opt = torch.optim.Adam(abl2_disc.parameters(), lr=CFG.LR*0.5, betas=(0.5, 0.9))

GAN_LAMBDA = 0.1 
window = torch.hann_window(CFG.N_FFT).to(CFG.DEVICE)
abl2_history = {'train_g_loss': [], 'train_d_loss': [], 'val_loss': [], 'val_sisdr': []}

print(f"GAN ablation ready | λ_adv = {GAN_LAMBDA}")

In [ ]:
for epoch in range(CFG.EPOCHS):
    abl2_gen.train(); abl2_disc.train()
    total_g, total_d = 0, 0

    for noisy, clean in abl2_train_loader:
        noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)

        with torch.no_grad():
            pred_wave, _ = abl2_gen(noisy)
        real_score = abl2_disc(clean)
        fake_score = abl2_disc(pred_wave.detach())
        d_loss = F.relu(1.0 - real_score).mean() + F.relu(1.0 + fake_score).mean()
        abl2_disc_opt.zero_grad()
        d_loss.backward()
        abl2_disc_opt.step()

        pred_wave, pred_mag = abl2_gen(noisy)
        clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                               window=window, return_complex=True).abs()
        l1_loss  = F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)
        adv_loss = -abl2_disc(pred_wave).mean()         
        g_loss   = l1_loss + GAN_LAMBDA * adv_loss
        abl2_gen_opt.zero_grad(); g_loss.backward()
        nn.utils.clip_grad_norm_(abl2_gen.parameters(), 5.0)
        abl2_gen_opt.step()

        total_g += g_loss.item(); total_d += d_loss.item()

    abl2_gen.eval()
    total_val, total_sisdr = 0, 0
    with torch.no_grad():
        for noisy, clean in abl2_val_loader:
            noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
            pred_wave, pred_mag = abl2_gen(noisy)
            clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                                   window=window, return_complex=True).abs()
            total_val   += (F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)).item()
            total_sisdr += si_sdr(pred_wave, clean).item()

    avg_g  = total_g   / len(abl2_train_loader)
    avg_d  = total_d   / len(abl2_train_loader)
    avg_vl = total_val / len(abl2_val_loader)
    avg_si = total_sisdr / len(abl2_val_loader)
    abl2_history['train_g_loss'].append(avg_g)
    abl2_history['train_d_loss'].append(avg_d)
    abl2_history['val_loss'].append(avg_vl)
    abl2_history['val_sisdr'].append(avg_si)
    print(f"[GAN] Epoch {epoch+1:02d} | G: {avg_g:.4f} | D: {avg_d:.4f} | Val: {avg_vl:.4f} | SI-SDR: {avg_si:.2f} dB")

torch.save(abl2_gen.state_dict(),  "abl2_gan_gen.pth")
torch.save(abl2_disc.state_dict(), "abl2_gan_disc.pth")
print("Saved abl2_gan_gen.pth + abl2_gan_disc.pth")

In [ ]:
abl2_gen.eval()
preview_noisy, preview_clean = next(iter(abl2_val_loader))
preview_noisy = preview_noisy[:1].to(CFG.DEVICE)
with torch.no_grad():
    enh, _ = abl2_gen(preview_noisy)
preview_audio(preview_noisy[0], enh[0], preview_clean[0])
preview_visuals(preview_noisy[0], enh[0].cpu(), preview_clean[0])

# ablation 3 : doubled sequence length
Doubled segment length from 32k to 64k, it is similar to remove cropping a bit, but mainly it's to see whether cropping with longer sequences in place will perform better than baseline or no cropping.

In [ ]:
SEGMENT_64K = 64000
BATCH_64K   = 4  

abl3_train_ds = ValentiniDataset(train_noisy, train_clean, segment=SEGMENT_64K, train=True)
abl3_val_ds   = ValentiniDataset(val_noisy,   val_clean,   segment=SEGMENT_64K, train=False)
abl3_train_loader = DataLoader(abl3_train_ds, batch_size=BATCH_64K, shuffle=True,  num_workers=2)
abl3_val_loader   = DataLoader(abl3_val_ds,   batch_size=BATCH_64K, shuffle=False, num_workers=2)

abl3_model = Denoiser().to(CFG.DEVICE)
abl3_opt   = torch.optim.Adam(abl3_model.parameters(), lr=CFG.LR)
window     = torch.hann_window(CFG.N_FFT).to(CFG.DEVICE)
abl3_history = {'train_loss': [], 'val_loss': [], 'val_sisdr': []}

print(f"Seg64k ablation | segment={SEGMENT_64K} | batch={BATCH_64K}")
print(f"Train batches: {len(abl3_train_loader)} | Val batches: {len(abl3_val_loader)}")

In [ ]:
for epoch in range(CFG.EPOCHS):
    abl3_model.train()
    total_train = 0
    for noisy, clean in abl3_train_loader:
        noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
        abl3_opt.zero_grad()
        pred_wave, pred_mag = abl3_model(noisy)
        clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                               window=window, return_complex=True).abs()
        loss = F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)
        loss.backward()
        nn.utils.clip_grad_norm_(abl3_model.parameters(), 5.0)
        abl3_opt.step()
        total_train += loss.item()

    abl3_model.eval()
    total_val, total_sisdr = 0, 0
    with torch.no_grad():
        for noisy, clean in abl3_val_loader:
            noisy, clean = noisy.to(CFG.DEVICE), clean.to(CFG.DEVICE)
            pred_wave, pred_mag = abl3_model(noisy)
            clean_mag = torch.stft(clean, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
                                   window=window, return_complex=True).abs()
            total_val   += (F.l1_loss(pred_mag, clean_mag) + 0.2 * F.l1_loss(pred_wave, clean)).item()
            total_sisdr += si_sdr(pred_wave, clean).item()

    avg_tr = total_train / len(abl3_train_loader)
    avg_vl = total_val   / len(abl3_val_loader)
    avg_si = total_sisdr / len(abl3_val_loader)
    abl3_history['train_loss'].append(avg_tr)
    abl3_history['val_loss'].append(avg_vl)
    abl3_history['val_sisdr'].append(avg_si)
    print(f"[Seg64k] Epoch {epoch+1} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f} | SI-SDR: {avg_si:.2f} dB")

torch.save(abl3_model.state_dict(), "abl3_seg64k.pth")
print("Saved abl3_seg64k.pth")

In [ ]:
abl3_model.eval()
preview_noisy, preview_clean = next(iter(abl3_val_loader))
preview_noisy = preview_noisy[:1].to(CFG.DEVICE)
with torch.no_grad():
    enh, _ = abl3_model(preview_noisy)
preview_audio(preview_noisy[0], enh[0], preview_clean[0])
preview_visuals(preview_noisy[0], enh[0].cpu(), preview_clean[0])

## results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

runs = [
    ('Baseline', base_history, 'D', 'black'),
    ('NoCrop',   abl1_history, 'o', None),
    ('GAN',      abl2_history, 's', None),
    ('Seg64k',   abl3_history, '^', None),
]

for name, h, mk, col in runs:
    kw = dict(marker=mk, label=name)
    if col: 
        kw['color'] = col
    axes[0].plot(range(1, len(h['val_loss'])  + 1), h['val_loss'],  **kw)
    axes[1].plot(range(1, len(h['val_sisdr']) + 1), h['val_sisdr'], **kw)

axes[0].set_title('Validation L1 Loss');    axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Validation SI-SDR (dB)'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

print("\n=== Final Epoch Results (@ epoch 2) ===")
print(f"{'Run':<12} {'Val Loss':>10} {'SI-SDR (dB)':>12}")
print("-" * 36)
for name, h, _, _ in runs:
    print(f"{name:<12} {h['val_loss'][-1]:>10.4f} {h['val_sisdr'][-1]:>12.2f}")